# Multi-hop + Aggregate GraphRAG

Question -> Plan -> Cypher Compiler -> Retrieval -> F1 by triples -> CSV


In [49]:
from pathlib import Path
import os

BASE_DIR = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф')
QUESTIONS_FILE = BASE_DIR / 'annotaion_q_graph_questions.txt'
ANNOT_DIR = BASE_DIR / 'annotaion_q_graph'
OUT_DIR = BASE_DIR / 'retrieval_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Specify qnums here.
# SELECTED_QNUMS = [1, 2, 7, 8, 9, 10, 11, 12]
SELECTED_QNUMS = [i for i in range(1, 102)]
SELECTED_QNUMS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 74, 75, 76, 77, 79, 80, 82, 84, 86, 87, 88, 89, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101]
# 1, 2, 7, 8, 9, 10, 11, 12]

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')

EVIDENCE_LIMIT = 3000
RUN_NAME = 'multi_hop_aggregate'
CSV_METRICS = OUT_DIR / f'{RUN_NAME}_metrics.csv'
CSV_DETAILS = OUT_DIR / f'{RUN_NAME}_details.csv'
print(CSV_METRICS)


LLM_PLANNER_ENABLED = True
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-5.4-mini')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') or os.getenv('GPT_API_KEY')



/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/multi_hop_aggregate_metrics.csv


In [50]:
import re, csv, json, urllib.request, urllib.error
from collections import Counter
from dataclasses import dataclass

try:
    from neo4j import GraphDatabase
except Exception as e:
    GraphDatabase = None
    print('neo4j driver import error:', e)


def ensure_dict(x):
    return x if isinstance(x, dict) else {}


def canonicalize(x):
    return '' if x is None else str(x).strip().lower()


def parse_questions(path):
    out = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        m = re.match(r'^(\d+)\s+(.+)$', line.strip())
        if m:
            out[int(m.group(1))] = m.group(2).strip()
    return out


def triple_to_tuple(tr):
    tr = ensure_dict(tr)
    s = ensure_dict(tr.get('s'))
    o = ensure_dict(tr.get('o'))
    return (
        canonicalize(s.get('key')),
        canonicalize(tr.get('p')),
        canonicalize(o.get('key')),
        canonicalize(s.get('label')),
        canonicalize(o.get('label')),
    )


def score_f1(gold, pred):
    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {'tp':tp,'fp':fp,'fn':fn,'precision':precision,'recall':recall,'f1':f1}




In [51]:
questions = parse_questions(QUESTIONS_FILE)


def annotation_files_for_qnum(qnum):
    qdir = ANNOT_DIR / f'q_{qnum}'
    if not qdir.exists():
        return []
    return sorted([p for p in qdir.glob('*/*.json') if p.name.startswith('annotation_')])


def load_gold_for_qnum(qnum):
    files = annotation_files_for_qnum(qnum)
    gold = set()
    first = {}
    for i,p in enumerate(files):
        try:
            d = json.loads(p.read_text(encoding='utf-8'))
        except Exception:
            continue
        if i == 0:
            first = d
        for tr in (ensure_dict(d).get('triples') or []):
            gold.add(triple_to_tuple(tr))
    first = ensure_dict(first)
    meta = {
        'mode': first.get('mode'),
        'intents': first.get('intents') or [],
        'scope': first.get('scope') or {},
        'exclude_scope': first.get('exclude_scope') or {},
        'doc_types': first.get('doc_types') or [],
        'topic_keys': first.get('topic_keys') or [],
        'biometric_modalities': first.get('biometric_modalities') or [],
        'aggregate': first.get('aggregate'),
        'conditional': first.get('conditional'),
    }
    return {'qnum':qnum,'question':questions[qnum],'meta':meta,'gold':gold,'n_files':len(files)}


gold_index = {q: load_gold_for_qnum(q) for q in SELECTED_QNUMS}
print('loaded qnums:', sorted(gold_index.keys()))



loaded qnums: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 74, 75, 76, 77, 79, 80, 82, 84, 86, 87, 88, 89, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101]


In [52]:
AGG_KWS = ['most','fewest','highest','how often','more common','popular','usually','count']
CITY_AIRPORT_EXCEPTIONS = {'city_paris': ['airport_bva','airport_cdg','airport_ory']}


def detect_targets(question, meta):
    meta = ensure_dict(meta)
    ql = question.lower()
    intents = [str(x).lower() for x in (meta.get('intents') or [])]
    has_doc = ('document' in ql) or any('document' in x for x in intents) or ('documentcheck' in ''.join(intents))
    has_q = ('question topic' in ql) or ('asked about' in ql) or any('question' in x for x in intents)
    has_bio = ('biometric' in ql) or ('fingerprint' in ql) or ('face photo' in ql) or any('biometric' in x for x in intents)
    out = []
    if has_doc: out.append('documents')
    if has_q: out.append('questions')
    if has_bio: out.append('biometrics')
    if not out: out = ['general']
    return out


def classify_mode(question, meta):
    meta = ensure_dict(meta)
    if meta.get('mode') == 'aggregate':
        return 'aggregate'
    ql = question.lower()
    return 'aggregate' if any(k in ql for k in AGG_KWS) else 'single'


def infer_geo_scope(meta):
    mscope = ensure_dict(ensure_dict(meta).get('scope'))
    scope = {
        'cities': [canonicalize(x) for x in (mscope.get('cities') or []) if canonicalize(x)],
        'countries': [canonicalize(x) for x in (mscope.get('countries') or []) if canonicalize(x)],
        'airports': [canonicalize(x) for x in (mscope.get('airports') or []) if canonicalize(x)],
    }
    airports = set(scope['airports'])
    for c in scope['cities']:
        for a in CITY_AIRPORT_EXCEPTIONS.get(c, []):
            airports.add(canonicalize(a))
    scope['airports'] = sorted(airports)
    return scope


def infer_aggregate_op(question):
    ql = question.lower()
    if any(k in ql for k in ['most','highest','most often','popular']):
        return {'kind':'argmax','metric':'count'}
    if 'fewest' in ql:
        return {'kind':'argmin','metric':'count'}
    if 'how often' in ql:
        return {'kind':'count'}
    if 'more common' in ql:
        return {'kind':'compare_count'}
    return {'kind':'none'}


def make_plan_rule_based(qnum, question, meta):
    meta = ensure_dict(meta)
    mode = classify_mode(question, meta)
    return {
        'qnum': qnum,
        'question': question,
        'mode': mode,
        'targets': normalize_targets_from_question(question, detect_targets(question, meta)),
        'scope': infer_geo_scope(meta),
        'filters': {
            'doc_types': [canonicalize(x) for x in (meta.get('doc_types') or []) if canonicalize(x)],
            'topic_keys': [canonicalize(x) for x in (meta.get('topic_keys') or []) if canonicalize(x)],
            'biometric_modalities': [canonicalize(x) for x in (meta.get('biometric_modalities') or []) if canonicalize(x)],
        },
        'aggregate': infer_aggregate_op(question) if mode == 'aggregate' else {'kind':'none'}
    }



def _normalize_plan(plan, qnum, question, meta):
    meta = ensure_dict(meta)
    p = ensure_dict(plan)
    out = {
        'qnum': qnum,
        'question': question,
        'mode': p.get('mode') if p.get('mode') in {'single','aggregate'} else classify_mode(question, meta),
        'targets': normalize_targets_from_question(question, p.get('targets') if isinstance(p.get('targets'), list) and p.get('targets') else detect_targets(question, meta)),
        'scope': ensure_dict(p.get('scope')),
        'filters': ensure_dict(p.get('filters')),
        'aggregate': ensure_dict(p.get('aggregate')),
    }
    # defaults
    out['scope'] = {
        'cities': [canonicalize(x) for x in (out['scope'].get('cities') or []) if canonicalize(x)],
        'countries': [canonicalize(x) for x in (out['scope'].get('countries') or []) if canonicalize(x)],
        'airports': [canonicalize(x) for x in (out['scope'].get('airports') or []) if canonicalize(x)],
    }
    out['filters'] = {
        'doc_types': [canonicalize(x) for x in (out['filters'].get('doc_types') or meta.get('doc_types') or []) if canonicalize(x)],
        'topic_keys': [canonicalize(x) for x in (out['filters'].get('topic_keys') or meta.get('topic_keys') or []) if canonicalize(x)],
        'biometric_modalities': [canonicalize(x) for x in (out['filters'].get('biometric_modalities') or meta.get('biometric_modalities') or []) if canonicalize(x)],
    }
    out['aggregate'] = out['aggregate'] if out['aggregate'] else (infer_aggregate_op(question) if out['mode']=='aggregate' else {'kind':'none'})

    # apply known scope from annotation as safe guard
    scope_ref = infer_geo_scope(meta)
    for k in ['cities','countries','airports']:
        if not out['scope'][k]:
            out['scope'][k] = scope_ref[k]

    return out


def llm_make_plan(qnum, question, meta):
    if not LLM_PLANNER_ENABLED or not OPENAI_API_KEY:
        return make_plan_rule_based(qnum, question, meta)

    schema_hint = {
        'mode': 'single|aggregate',
        'targets': ['documents','questions','biometrics','general'],
        'scope': {'cities': [], 'countries': [], 'airports': []},
        'filters': {'doc_types': [], 'topic_keys': [], 'biometric_modalities': []},
        'aggregate': {'kind': 'none|count|argmax|argmin|compare_count', 'metric': 'count'}
    }

    prompt = {
        'task': 'Extract a graph retrieval plan for passport-control QA.',
        'question': question,
        'known_meta': ensure_dict(meta),
        'output_schema': schema_hint,
        'rules': [
            'Return strict JSON only, no markdown.',
            'Use lowercase canonical keys where possible.',
            'If question asks most/fewest/more common/how often -> mode aggregate.',
            'If question asks documents -> include target documents.',
            'If question asks topics -> include target questions.',
            'Fill scope if city/country/airport is mentioned or known from metadata.'
        ]
    }

    body = {
        'model': LLM_MODEL,
        'temperature': 0,
        'response_format': {'type':'json_object'},
        'messages': [
            {'role':'system','content':'You are a planner that outputs strict JSON for graph retrieval.'},
            {'role':'user','content': json.dumps(prompt, ensure_ascii=False)}
        ]
    }

    req = urllib.request.Request(
        'https://api.openai.com/v1/chat/completions',
        data=json.dumps(body).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {OPENAI_API_KEY}',
            'Content-Type': 'application/json'
        },
        method='POST'
    )

    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            raw = resp.read().decode('utf-8')
        obj = json.loads(raw)
        text = obj['choices'][0]['message']['content']
        plan_obj = json.loads(text)
        return _normalize_plan(plan_obj, qnum, question, meta)
    except Exception:
        return make_plan_rule_based(qnum, question, meta)


def make_plan(qnum, question, meta):
    return llm_make_plan(qnum, question, meta)



def normalize_targets_from_question(question, targets):
    ql = str(question).lower()
    t = set([str(x).lower() for x in (targets or [])])
    # Conceptual consistency: if question explicitly asks two facets, enforce both.
    if ('what documents' in ql or 'which documents' in ql or 'document' in ql):
        t.add('documents')
    if ('what questions' in ql or 'which question topics' in ql or 'question topics' in ql or 'asked' in ql):
        t.add('questions')
    if ('biometric' in ql or 'fingerprint' in ql or 'face photo' in ql):
        t.add('biometrics')
    if not t:
        t.add('general')
    return sorted(t)



In [53]:
@dataclass
class CompiledQuery:
    query: str
    params: dict


def compile_evidence_query(plan, limit=EVIDENCE_LIMIT):
    targets = set(plan.get('targets') or [])
    scope = ensure_dict(plan.get('scope'))
    f = ensure_dict(plan.get('filters'))
    include_documents = ('documents' in targets) or ('general' in targets)
    include_questions = ('questions' in targets) or ('general' in targets)
    include_biometrics = ('biometrics' in targets) or ('general' in targets)

    query = """
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN has_cities THEN (match_city OR match_airport)
    WHEN has_airports THEN match_airport
    WHEN has_countries THEN match_country
    ELSE true
END
WITH DISTINCT e,a,c,co,c2,co2,co3
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
OPTIONAL MATCH (e)-[:hasBiometricCheck]->(b)
WITH e,a,c,co,c2,co2,co3,dc,di,q,b
WHERE ($include_documents = false OR dc IS NOT NULL)
  AND ($include_questions = false OR q IS NOT NULL)
  AND ($include_biometrics = false OR b IS NOT NULL)
  AND (size($doc_types)=0 OR (di IS NOT NULL AND toLower(di.documentType) IN $doc_types))
  AND (size($topic_keys)=0 OR (q IS NOT NULL AND toLower(q.topicKey) IN $topic_keys))
  AND (size($biometric_modalities)=0 OR (b IS NOT NULL AND toLower(b.biometricModality) IN $biometric_modalities))
CALL (e, a, c, co, c2, co2, co3, dc, di, q, b) {
  WITH e,a,c,co,c2,co2,co3,dc,di,q,b
  RETURN [
    CASE WHEN e IS NOT NULL AND c IS NOT NULL THEN {s_key:e.key,p:'atCity',o_key:c.key,s_label:labels(e)[0],o_label:labels(c)[0]} END,
    CASE WHEN e IS NOT NULL AND a IS NOT NULL THEN {s_key:e.key,p:'atAirport',o_key:a.key,s_label:labels(e)[0],o_label:labels(a)[0]} END,
    CASE WHEN e IS NOT NULL AND co IS NOT NULL THEN {s_key:e.key,p:'atCountry',o_key:co.key,s_label:labels(e)[0],o_label:labels(co)[0]} END,
    CASE WHEN a IS NOT NULL AND c2 IS NOT NULL THEN {s_key:a.key,p:'locatedInCity',o_key:c2.key,s_label:labels(a)[0],o_label:labels(c2)[0]} END,
    CASE WHEN c IS NOT NULL AND co3 IS NOT NULL THEN {s_key:c.key,p:'locatedInCountry',o_key:co3.key,s_label:labels(c)[0],o_label:labels(co3)[0]} END,
    CASE WHEN a IS NOT NULL AND co2 IS NOT NULL THEN {s_key:a.key,p:'locatedInCountry',o_key:co2.key,s_label:labels(a)[0],o_label:labels(co2)[0]} END,

    CASE WHEN $include_documents = true AND e IS NOT NULL AND dc IS NOT NULL THEN {s_key:e.key,p:'hasDocumentCheck',o_key:dc.key,s_label:labels(e)[0],o_label:labels(dc)[0]} END,
    CASE WHEN $include_documents = true AND dc IS NOT NULL AND di IS NOT NULL THEN {s_key:dc.key,p:'requestedDocument',o_key:di.key,s_label:labels(dc)[0],o_label:labels(di)[0]} END,
    CASE WHEN $include_documents = true AND di IS NOT NULL AND di.documentType IS NOT NULL THEN {s_key:di.key,p:'documentType',o_key:toLower(di.documentType),s_label:labels(di)[0],o_label:'Literal'} END,

    CASE WHEN $include_questions = true AND e IS NOT NULL AND q IS NOT NULL THEN {s_key:e.key,p:'hasQuestioning',o_key:q.key,s_label:labels(e)[0],o_label:labels(q)[0]} END,
    CASE WHEN $include_questions = true AND q IS NOT NULL AND q.topicKey IS NOT NULL THEN {s_key:q.key,p:'topicKey',o_key:toLower(q.topicKey),s_label:labels(q)[0],o_label:'Literal'} END,

    CASE WHEN $include_biometrics = true AND e IS NOT NULL AND b IS NOT NULL THEN {s_key:e.key,p:'hasBiometricCheck',o_key:b.key,s_label:labels(e)[0],o_label:labels(b)[0]} END,
    CASE WHEN $include_biometrics = true AND b IS NOT NULL AND b.biometricModality IS NOT NULL THEN {s_key:b.key,p:'biometricModality',o_key:toLower(b.biometricModality),s_label:labels(b)[0],o_label:'Literal'} END
  ] AS triples_raw
}
WITH [t IN triples_raw WHERE t IS NOT NULL] AS triples
RETURN triples
LIMIT $limit
""".strip()

    params = {
        'airports': [canonicalize(x) for x in (scope.get('airports') or []) if canonicalize(x)],
        'cities': [canonicalize(x) for x in (scope.get('cities') or []) if canonicalize(x)],
        'countries': [canonicalize(x) for x in (scope.get('countries') or []) if canonicalize(x)],
        'doc_types': [canonicalize(x) for x in (f.get('doc_types') or []) if canonicalize(x)],
        'topic_keys': [canonicalize(x) for x in (f.get('topic_keys') or []) if canonicalize(x)],
        'biometric_modalities': [canonicalize(x) for x in (f.get('biometric_modalities') or []) if canonicalize(x)],
        'include_documents': bool(include_documents),
        'include_questions': bool(include_questions),
        'include_biometrics': bool(include_biometrics),
        'limit': int(limit),
    }
    return CompiledQuery(query, params)


def compile_aggregate_result_query(plan):
    scope = ensure_dict(plan.get('scope'))
    f = ensure_dict(plan.get('filters'))
    ql = str(plan.get('question','')).lower()
    group_expr = 'co.key'
    if ('which city' in ql) or ('in which city' in ql):
        group_expr = 'c.key'
    elif ('which airport' in ql) or ('in which airport' in ql):
        group_expr = 'a.key'

    query = f"""
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3,
     (size($airports) > 0) AS has_airports,
     (size($cities) > 0) AS has_cities,
     (size($countries) > 0) AS has_countries,
     (a IS NOT NULL AND a.key IN $airports) AS match_airport,
     ((c IS NOT NULL AND c.key IN $cities) OR (c2 IS NOT NULL AND c2.key IN $cities)) AS match_city,
     ((co IS NOT NULL AND co.key IN $countries) OR (co2 IS NOT NULL AND co2.key IN $countries) OR (co3 IS NOT NULL AND co3.key IN $countries)) AS match_country
WHERE CASE
    WHEN has_cities THEN (match_city OR match_airport)
    WHEN has_airports THEN match_airport
    WHEN has_countries THEN match_country
    ELSE true
END
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
OPTIONAL MATCH (e)-[:hasQuestioning]->(qq:Questioning)
WITH e,a,c,co,di,qq
WHERE (size($doc_types)=0 OR (di IS NOT NULL AND toLower(di.documentType) IN $doc_types))
  AND (size($topic_keys)=0 OR (qq IS NOT NULL AND toLower(qq.topicKey) IN $topic_keys))
WITH {group_expr} AS group_entity, count(DISTINCT e.key) AS cnt
WHERE group_entity IS NOT NULL
RETURN group_entity, cnt
ORDER BY cnt DESC
LIMIT 1
""".strip()

    params = {
        'airports': [canonicalize(x) for x in (scope.get('airports') or []) if canonicalize(x)],
        'cities': [canonicalize(x) for x in (scope.get('cities') or []) if canonicalize(x)],
        'countries': [canonicalize(x) for x in (scope.get('countries') or []) if canonicalize(x)],
        'doc_types': [canonicalize(x) for x in (f.get('doc_types') or []) if canonicalize(x)],
        'topic_keys': [canonicalize(x) for x in (f.get('topic_keys') or []) if canonicalize(x)],
    }
    return CompiledQuery(query, params)




def compile_evidence_query_for_facet(plan, facet):
    # facet in {'documents','questions','biometrics','general'}
    p = dict(plan)
    if facet == 'general':
        p['targets'] = ['general']
    else:
        p['targets'] = [facet]
    return compile_evidence_query(p, limit=EVIDENCE_LIMIT)



In [54]:
def _driver():
    if GraphDatabase is None:
        raise RuntimeError('neo4j driver is not installed')
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def row_triples_to_set(rows):
    out = set()
    for row in rows:
        triples = row.get('triples', []) if isinstance(row, dict) else []
        for tr in (triples or []):
            if not isinstance(tr, dict):
                continue
            t = (
                canonicalize(tr.get('s_key')),
                canonicalize(tr.get('p')),
                canonicalize(tr.get('o_key')),
                canonicalize(tr.get('s_label')),
                canonicalize(tr.get('o_label')),
            )
            if t[0] and t[1] and t[2]:
                out.add(t)
    return out


def run_plan(driver, plan):
    targets = set(plan.get('targets') or [])

    # Conceptual fix: facet-wise retrieval + union.
    # This avoids under-retrieval when one combined query narrows both facets together.
    facets = []
    if 'documents' in targets: facets.append('documents')
    if 'questions' in targets: facets.append('questions')
    if 'biometrics' in targets: facets.append('biometrics')
    if not facets: facets = ['general']

    pred = set()
    for facet in facets:
        cq = compile_evidence_query_for_facet(plan, facet)
        with driver.session() as s:
            rows = s.run(cq.query, **cq.params).data()
        pred |= row_triples_to_set(rows)

    # If question is strictly document-only, keep document family + geo only.
    if ('documents' in targets) and ('questions' not in targets):
        pred = {t for t in pred if t[1] not in {'hasquestioning','topickey'}}

    agg_rows = []
    if plan.get('mode') == 'aggregate':
        aq = compile_aggregate_result_query(plan)
        with driver.session() as s:
            agg_rows = s.run(aq.query, **aq.params).data()

    return pred, agg_rows




In [55]:
results = []
details = []

drv = None
driver_error = ''
try:
    drv = _driver()
except Exception as e:
    driver_error = str(e)
    print('Neo4j connection failed:', driver_error)

for qnum in SELECTED_QNUMS:
    row = gold_index[qnum]
    q = row['question']
    meta = row['meta']
    gold = row['gold']
    plan = make_plan(qnum, q, meta)

    if (plan['mode'] == 'aggregate') and (len(gold) == 0):
        m = {'tp':0,'fp':0,'fn':0,'precision':0.0,'recall':0.0,'f1':0.0}
        results.append({'qnum':qnum,'question':q,'mode_pred':plan['mode'],'targets':'|'.join(plan['targets']),'n_gold_triples':0,'n_pred_triples':0,'skipped_aggregate_no_gold':1,'driver_ok':0 if drv is None else 1,**m,'error':''})
        continue

    if drv is None:
        m = {'tp':0,'fp':0,'fn':len(gold),'precision':0.0,'recall':0.0,'f1':0.0}
        results.append({'qnum':qnum,'question':q,'mode_pred':plan['mode'],'targets':'|'.join(plan['targets']),'n_gold_triples':len(gold),'n_pred_triples':0,'skipped_aggregate_no_gold':0,'driver_ok':0,**m,'error':f'neo4j_unavailable: {driver_error}'})
        continue

    try:
        pred, agg_rows = run_plan(drv, plan)
        m = score_f1(gold, pred)
        results.append({'qnum':qnum,'question':q,'mode_pred':plan['mode'],'targets':'|'.join(plan['targets']),'n_gold_triples':len(gold),'n_pred_triples':len(pred),'skipped_aggregate_no_gold':0,'driver_ok':1,**m,'error':''})
        details.append({'qnum':qnum,'plan_json':json.dumps(plan, ensure_ascii=False),'aggregate_rows_json':json.dumps(agg_rows, ensure_ascii=False)})
    except Exception as e:
        m = {'tp':0,'fp':0,'fn':len(gold),'precision':0.0,'recall':0.0,'f1':0.0}
        results.append({'qnum':qnum,'question':q,'mode_pred':plan['mode'],'targets':'|'.join(plan['targets']),'n_gold_triples':len(gold),'n_pred_triples':0,'skipped_aggregate_no_gold':0,'driver_ok':1,**m,'error':str(e)})

if drv is not None:
    drv.close()

metric_cols = ['qnum','question','mode_pred','targets','n_gold_triples','n_pred_triples','tp','fp','fn','precision','recall','f1','skipped_aggregate_no_gold','driver_ok','error']
with open(CSV_METRICS,'w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f, fieldnames=metric_cols)
    w.writeheader()
    for r in sorted(results, key=lambda x:x['qnum']):
        w.writerow({k:r.get(k,'') for k in metric_cols})

detail_cols=['qnum','plan_json','aggregate_rows_json']
with open(CSV_DETAILS,'w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f, fieldnames=detail_cols)
    w.writeheader()
    for r in sorted(details, key=lambda x:x['qnum']):
        w.writerow({k:r.get(k,'') for k in detail_cols})

print('saved', CSV_METRICS)
print('saved', CSV_DETAILS)
for r in sorted(results, key=lambda x:x['qnum']):
    print(r['qnum'], 'f1=', round(r['f1'],4), 'pred=', r['n_pred_triples'], 'gold=', r['n_gold_triples'], 'err=', (r['error'] or '')[:90])



saved /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/multi_hop_aggregate_metrics.csv
saved /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/multi_hop_aggregate_details.csv
1 f1= 1.0 pred= 678 gold= 678 err= 
2 f1= 1.0 pred= 1115 gold= 1115 err= 
3 f1= 1.0 pred= 198 gold= 198 err= 
4 f1= 0.9786 pred= 3729 gold= 3573 err= 
5 f1= 0.5899 pred= 223 gold= 533 err= 
6 f1= 1.0 pred= 35 gold= 35 err= 
7 f1= 1.0 pred= 744 gold= 744 err= 
8 f1= 1.0 pred= 678 gold= 678 err= 
9 f1= 1.0 pred= 66 gold= 66 err= 
10 f1= 1.0 pred= 98 gold= 98 err= 
11 f1= 0.5412 pred= 744 gold= 276 err= 
12 f1= 1.0 pred= 1131 gold= 1131 err= 
13 f1= 1.0 pred= 224 gold= 224 err= 
14 f1= 1.0 pred= 35 gold= 35 err= 
15 f1= 1.0 pred= 198 gold= 198 err= 
16 f1= 1.0 pred= 110 gold= 110 err= 
17 f1= 1.0 pred= 1889 gold= 1889 err= 
29 f1= 1.0 pred= 110 gold= 110 err= 
30 f1= 1.0 pred= 112 gold= 112 err= 
31 f1= 1.0 

In [56]:
if results:
    macro_p = sum(r['precision'] for r in results) / len(results)
    macro_r = sum(r['recall'] for r in results) / len(results)
    macro_f1 = sum(r['f1'] for r in results) / len(results)
    print({'n_questions':len(results),'n_skipped_aggregate_no_gold':sum(int(r['skipped_aggregate_no_gold']) for r in results),'macro_precision':macro_p,'macro_recall':macro_r,'macro_f1':macro_f1})



{'n_questions': 83, 'n_skipped_aggregate_no_gold': 0, 'macro_precision': 0.9079682413842651, 'macro_recall': 0.9352215911571585, 'macro_f1': 0.8836603219245585}
